 # 03 · Validación Técnica Bronze — YelpConnect

 **Tipo:** Quality Gate técnico de ingesta  
 **Capa:** Bronze  
 **Alcance:** Business, User, Tip, Check-in, Review

 ---

 ## Propósito

 Validar que RAW → Bronze produjo las cinco tablas esperadas con
 metadata técnica mínima y contenido consultable.

 Este gate valida la **ingesta técnica**.
 No sustituye la ejecución formal de reglas Data Quality.

In [0]:
# COMMAND ----------

import os
import sys
import json
import importlib.util

from pyspark.sql import functions as F


# ============================================================
# 1. CARGAR MÓDULO COMÚN BRONZE
# ============================================================

COMMON_MODULE_NAME = "bronze_common"

COMMON_MODULE_PATH = os.path.join(
    os.getcwd(),
    "bronze_common.py"
)


print("=" * 72)
print("YELPCONNECT - VALIDACIÓN TÉCNICA BRONZE")
print("=" * 72)

print(
    f"Working directory : {os.getcwd()}"
)

print(
    f"Módulo común      : {COMMON_MODULE_PATH}"
)


if not os.path.isfile(
    COMMON_MODULE_PATH
):

    raise FileNotFoundError(
        f"""
        ❌ No se encontró el módulo común Bronze.

        Ruta esperada:
        {COMMON_MODULE_PATH}
        """
    )


spec = importlib.util.spec_from_file_location(
    COMMON_MODULE_NAME,
    COMMON_MODULE_PATH
)


if (
    spec is None
    or spec.loader is None
):

    raise RuntimeError(
        f"""
        ❌ No fue posible crear el loader para:

        {COMMON_MODULE_PATH}
        """
    )


bronze_common = (
    importlib.util.module_from_spec(
        spec
    )
)


sys.modules[
    COMMON_MODULE_NAME
] = bronze_common


spec.loader.exec_module(
    bronze_common
)


print(
    "✔ bronze_common.py cargado correctamente."
)


# ============================================================
# 2. EXPONER CONFIGURACIÓN COMPARTIDA
# ============================================================

CATALOG = bronze_common.CATALOG

BRONZE_SCHEMA = (
    bronze_common.BRONZE_SCHEMA
)

RAW = bronze_common.RAW

SOURCE_SYSTEM = (
    bronze_common.SOURCE_SYSTEM
)

LAYER = bronze_common.LAYER

METADATA_COLUMNS = (
    bronze_common.METADATA_COLUMNS
)


# ============================================================
# 3. EXPONER FUNCIONES COMPARTIDAS
# ============================================================

assert_bronze_schema_exists = (
    bronze_common
    .assert_bronze_schema_exists
)


# ============================================================
# 4. CONFIGURACIÓN DEL QUALITY GATE BRONZE
# ============================================================

PIPELINE_NAME = (
    "E7_01_validar_bronze"
)


EXPECTED_ENTITIES = {

    "brz_business":
        "business",

    "brz_user":
        "user",

    "brz_tip":
        "tip",

    "brz_checkin":
        "checkin",

    "brz_review":
        "review"
}


# ============================================================
# 5. VALIDAR PRECONDICIÓN
# ============================================================

full_schema = (
    assert_bronze_schema_exists(
        spark
    )
)


print()
print("=" * 72)
print("CONFIGURACIÓN QUALITY GATE BRONZE")
print("=" * 72)

print(
    f"Pipeline       : {PIPELINE_NAME}"
)

print(
    f"Catalog        : {CATALOG}"
)

print(
    f"Schema Bronze  : {BRONZE_SCHEMA}"
)

print(
    f"Schema validado: {full_schema}"
)

print(
    f"Source System  : {SOURCE_SYSTEM}"
)

print(
    f"Layer          : {LAYER}"
)

print(
    f"Tablas esperadas: "
    f"{len(EXPECTED_ENTITIES)}"
)

print("=" * 72)

 ## 1 · Validaciones

 Por tabla se verifica:

 - existencia;
 - cantidad mayor que cero;
 - presencia de metadata Bronze;
 - `_source_system = YELP`;
 - `_layer = bronze`;
 - `_source_entity` consistente;
 - checksum y schema hash informados.

 Cualquier incumplimiento detiene la etapa Bronze.


In [0]:
# COMMAND ----------

results = []
failed = []

for table_name, expected_entity in EXPECTED_ENTITIES.items():

    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    if not spark.catalog.tableExists(full_table):
        results.append({
            "table_name": table_name,
            "status": "FAIL",
            "total_records": 0,
            "metadata_ok": False,
            "detail": "Tabla inexistente"
        })
        failed.append(table_name)
        continue

    df = spark.table(full_table)
    total_records = df.count()

    missing_metadata = [
        column
        for column in METADATA_COLUMNS
        if column not in df.columns
    ]

    metadata_ok = len(missing_metadata) == 0
    detail_parts = []

    if total_records == 0:
        detail_parts.append("tabla sin registros")

    if missing_metadata:
        detail_parts.append(
            "metadata faltante=" + ",".join(missing_metadata)
        )

    semantic_errors = 0

    if metadata_ok and total_records > 0:
        semantic_errors = (
            df.filter(
                (F.col("_source_system") != SOURCE_SYSTEM)
                |
                (F.col("_layer") != LAYER)
                |
                (F.col("_source_entity") != expected_entity)
                |
                F.col("_record_checksum").isNull()
                |
                F.col("_schema_hash").isNull()
            )
            .count()
        )

        if semantic_errors > 0:
            detail_parts.append(
                f"metadata inconsistente={semantic_errors}"
            )

    status = (
        "PASS"
        if (
            total_records > 0
            and metadata_ok
            and semantic_errors == 0
        )
        else "FAIL"
    )

    if status == "FAIL":
        failed.append(table_name)

    results.append({
        "table_name": table_name,
        "status": status,
        "total_records": total_records,
        "metadata_ok": metadata_ok,
        "detail": (
            "; ".join(detail_parts)
            if detail_parts
            else "OK"
        )
    })

validation_df = (
    spark.createDataFrame(results)
    .orderBy("table_name")
)

display(validation_df)

# COMMAND ----------

 ## 2 · Quality Gate Bronze

 El gate técnico pasa únicamente cuando las cinco tablas tienen
 estado `PASS`.

In [0]:
# COMMAND ----------

if failed:
    raise RuntimeError(
        "Quality Gate técnico Bronze FALLÓ. Tablas: "
        + ", ".join(sorted(set(failed)))
    )

print("✔ Quality Gate técnico Bronze: PASS")

summary = {
    "status": "SUCCESS",
    "pipeline": PIPELINE_NAME,
    "tables": len(results),
    "passed": len(results),
    "failed": 0
}

dbutils.notebook.exit(json.dumps(summary))
